# Stock-Monitor: Financial Sentiment Class Project
This is the **new sentiment notebook**, not the old SQuAD question-answering notebook.

We ask: (A) can our model classify financial sentences? (B) does its output help an existing financial predictor?

Select **Runtime → Change runtime type → GPU**. Run each cell in order. If a cell errors, stop and share the error. Full metrics are only available after actual training. Read `class_project/START_HERE.md` in the fork for the longer explanation.


## 1. Save progress and load the class branch
Google Drive is optional but recommended: Colab's temporary disk can disappear. Mounting requires your Google consent. Your credentials are not sent to the project. A dedicated folder stores class artifacts only.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

USE_DRIVE = True  # Set False to use temporary storage; export before disconnecting.
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
repo = Path("/content/stock-monitor-class")
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "class/financial-sentiment-study",
            "https://github.com/AMurtadaa/Stock-Monitor.git",
            str(repo),
        ],
        check=True,
    )
os.chdir(repo)
if USE_DRIVE:
    saved = Path("/content/drive/MyDrive/CSCI390_Financial_Sentiment/class_artifacts")
    saved.mkdir(parents=True, exist_ok=True)
    local = repo / "class_artifacts"
    if not local.exists():
        local.symlink_to(saved, target_is_directory=True)
    elif local.resolve() != saved.resolve():
        raise RuntimeError(
            "Artifacts already exist outside Drive. Export them before changing storage."
        )
print("Working folder:", repo)
print("Artifacts:", (repo / "class_artifacts").resolve())

## 2. Isolated dependencies and GPU check
We use a separate Python environment that can access Colab's CUDA PyTorch. All experiment commands run through `run(...)`, which stops the cell on failure. Installation does not replace packages in your notebook kernel. First installation may take several minutes.


In [ ]:
env = Path("/content/stock-class-env")
if not env.exists():
    subprocess.run(
        [sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(env)],
        check=True,
    )
PYTHON = str(env / "bin/python")
# Colab may not provide ensurepip. Use the notebook's pip to bootstrap the target.
subprocess.run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "pip"], check=True)
subprocess.run([PYTHON, "-m", "pip", "--version"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-r", "class_project/requirements.txt"], check=True)
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def run(*args):
    subprocess.run([PYTHON, "-m", "class_project", *map(str, args)], check=True)


run("check")
subprocess.run(
    [PYTHON, "-c", "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime.'"],
    check=True,
)
subprocess.run([PYTHON, "-m", "pytest", "class_project/tests", "-q"], check=True)

## 3. Human-labeled financial sentences
Training examples teach the model; development examples select its checkpoint; test examples are the final exam. Preparation freezes and verifies the split. We preserve three named labels: negative, neutral and positive.


In [ ]:
run("prepare")
import json

manifest = json.loads(Path("class_artifacts/dataset/manifest.json").read_text())
print("Split sizes:", manifest["split_sizes"])
print("Class balance:", manifest["class_counts"])
print("Check: can you explain why test examples must not be used to choose a model?")

## 4. Smoke test first
This performs **two training steps**, then verifies saving, reloading and scoring on a small development sample. A low smoke-test score is expected. It is not a result to submit.


In [ ]:
run("train", "--smoke")
run("evaluate", "--smoke")

## 5. Real sentiment training
Three runs use identical data but different random seeds. Do not select the best test seed; seed 42 was selected in advance for the financial application. Each run shows progress and saves checkpoints. Completed runs are reused; interrupted runs resume from the last checkpoint.


In [ ]:
for seed in [42, 17, 73]:
    run("train", "--seed", seed)

## 6. Held-out evaluation
Accuracy means the fraction correct. Macro-F1 treats each class equally. FinBERT is a reference only because it saw Financial PhraseBank during its original training. Inspect the 15 review examples and add your explanations.


In [ ]:
run("evaluate", "--finbert")
metrics = json.loads(Path("class_artifacts/evaluation/full/metrics.json").read_text())
print(json.dumps(metrics["three_seed_summary"], indent=2))
from IPython.display import Image, display

display(Image(filename="class_artifacts/evaluation/full/seed-42_confusion.png"))
display(Image(filename="class_artifacts/evaluation/full/seed-42_curves.png"))

## 7. Historical news for the financial comparison
This is separate from sentiment training. We need actual timestamps and later prices to test whether sentiment helps predictions.

First inspect existing inputs. This fresh fork contains source, not your other machine's private database. If you already have the FNSPID archive, set `NEWS_CSV` to its path in Drive. Otherwise the default URL streams the public ~23 GB archive once, retaining only 12 stocks. **This can take a long time.** It prints progress; do not repeat after `headlines.csv` exists. No credentials or generated sentiment are used.


In [ ]:
run("inspect-data")
NEWS_CSV = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv"
# If you have the archive in Drive, replace the URL above with that file path.
if not Path("class_artifacts/financial/headlines.csv").exists():
    run("filter-news", "--csv", NEWS_CSV)
print(Path("class_artifacts/financial/news_manifest.json").read_text())
run("score-news")

## 8. Prices, filing-dated fundamentals, and the controlled comparison
SEC requires a descriptive contact User-Agent. Enter your own email only in the prompt; it is used for SEC requests and isn't saved in the class report. Free SEC requests may be blocked from some Colab networks; if so, stop and share the error so we can prepare the snapshot locally.

The comparison trains the same LightGBM model twice, changing only whether it receives sentiment. It stops if news coverage is too weak or dates are missing. It does not submit trades.


In [ ]:
from getpass import getpass

contact = getpass("SEC contact email (for request identification only): ").strip()
if not Path("class_artifacts/financial/snapshot/manifest.json").exists():
    run("snapshot", "--sec-user-agent", f"CSCI390Research/1.0 ({contact})")
del contact
run("compare")
print("Read coverage first:")
print(Path("class_artifacts/financial/coverage.json").read_text())

## 9. Prepare and export results before opening the demo
The report generator inserts actual saved metrics into a draft. It cannot write your personal interpretation for you. Download the archive before ending the session; Drive holds a second copy when enabled. Send `submission/results_summary.json` back for review. Do not send a private app database or API keys.


In [ ]:
run("writeup")
run("export")
from google.colab import files

files.download("class_artifacts/exports/class_training_artifacts.zip")

## 10. Interactive demonstration
This cell remains running while the demo is open. `--share` creates a temporary public link. Only curated examples and saved class metrics are shown. Try a positive, a negative, a neutral and a contrasting sentence. Stop this cell to close the demo.


In [ ]:
run("demo", "--finbert", "--share")